# **MODEL EXPERIMENTATION**
with GCP Integration


by Jack Phelan

In [75]:
#imports 
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)
from scripts.gcs_utils import (
    log_dataset_to_gcs,
    log_pipeline_run,
)
from kfp import compiler
from google.cloud import aiplatform

# Components
from vertex.components import (
    load_validate_data,
    split_data,
    oversample_training,
    fit_apply_preprocessing_v1,
    apply_preprocessing_v1,
    train_model,
    evaluate_model,
)

# Pipelines
from vertex.pipelines import preprocessing_pipeline, training_pipeline


In [76]:
# variable declarations
TARGET_COL = "readmission_within_30_days"
ID_COL = "patient_id"
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmissions_bucket_v2"


---
## KFP Preprocessing Pipeline

Self-contained KFP v2 components for the preprocessing pipeline.  
Order: `load_validate_data` → `split_data` → `oversample_training` → `fit_apply_preprocessing` → `apply_preprocessing`

In [77]:
# Components are defined in vertex/components/ and imported above:
#   load_validate_data                          ← ingest.py
#   split_data                                  ← split.py
#   oversample_training                         ← oversample.py
#   fit_apply_preprocessing_v1                  ← preprocessing.py
#   apply_preprocessing_v1                      ← preprocessing.py
#   train_model                                 ← train.py
#   evaluate_model                              ← evaluate.py


In [78]:
PREPROCESSING_PIPELINE_JSON = "../vertex/pipelines/readmissions_preprocessing_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=preprocessing_pipeline,
    package_path=PREPROCESSING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {PREPROCESSING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_preprocessing_pipeline.json


---
## KFP Training Pipeline

`train_model` and `evaluate_model` components extending the preprocessing pipeline.  
Order: `...preprocessing...` → `train_model` → `evaluate_model`

- **`model_type`**: `"logistic"` | `"random_forest"` | `"xgboost"`  
- **`hyperparams_json`**: JSON string of kwargs passed to the chosen estimator (e.g. `'{"n_estimators": 200, "max_depth": 5}'`)

In [79]:
TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path=TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {TRAINING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_training_pipeline.json


---
## Pipeline Submission with Vertex AI Datasets

Register the training CSV as a managed Vertex AI Dataset, then submit the training pipeline.  
The dataset resource name flows through `load_validate_data` metadata → Vertex ML Metadata, giving a native console lineage graph: **Dataset → Pipeline Run → Model**.

- To reuse an existing dataset instead of creating a new one, replace `TabularDataset.create(...)` with `aiplatform.TabularDataset(dataset_name="projects/.../datasets/...")`.

In [80]:
from pathlib import Path

RAW_TRAIN_PATH = Path("../data/raw/healthcare_readmissions_dataset_train.csv")
DATASET_VERSION = "v0.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Raw training data",
    tags=["raw"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/manifest.json


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v0-0


In [81]:
import time

# Register a Vertex AI Dataset for the current data version.
# This creates a managed dataset entry in Vertex AI linked to the GCS CSV.
DATASET_VERSION = "v0.0"
DATASET_GCS_URI = f"{BUCKET_ROOT_URI}/datasets/readmissions/{DATASET_VERSION}/train.csv"
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Reuse existing dataset if one already exists with this display name.
existing = aiplatform.TabularDataset.list(
    filter=f'display_name="readmissions-{DATASET_VERSION}"',
    order_by="create_time desc",
)

if existing:
    dataset = existing[0]
    print(f"Reusing existing dataset: {dataset.resource_name}")
else:
    # Create with simple retry for transient InternalServerError responses.
    for attempt in range(3):
        try:
            dataset = aiplatform.TabularDataset.create(
                display_name=f"readmissions-{DATASET_VERSION}",
                gcs_source=[DATASET_GCS_URI],
            )
            break
        except Exception as e:
            if attempt < 2:
                wait = 15 * (attempt + 1)
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

DATASET_RESOURCE_NAME = dataset.resource_name
print(f"Dataset registered : {DATASET_RESOURCE_NAME}")
print(f"GCS source         : {DATASET_GCS_URI}")


Reusing existing dataset: projects/182027088454/locations/us-central1/datasets/1513965389040582656
Dataset registered : projects/182027088454/locations/us-central1/datasets/1513965389040582656
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv


In [ ]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "readmissions-model-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "training_dataset_path": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "dataset_resource_name": DATASET_RESOURCE_NAME,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit(experiment=EXPERIMENT_NAME)
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,   # blocks until pipeline finishes and logs eval metrics
)


Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260512132741?project=182027088454
Associating projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 to Experiment: readmissions-model-exp
Pipeline submitted: readmissions-training-v0.0-v0


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/readmissions-model-exp-pipeline-run-v0-20260512202819 to Experiment: readmissions-model-exp


Waiting for pipeline to complete...
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
